# 02. 시계열 전처리와 데이터 분할

용해탱크 KPI MLOps v1 — Notebook 기반 모델 개발

## 전처리 원칙

- 입력 특성: `MELT_TEMP`, `MOTORSPEED`, `MELT_WEIGHT`
- 한 timestamp의 10개 행을 하나의 LSTM 시퀀스로 사용
- 현재 분 `X[t]`로 다음 분 `y[t+1]` 예측
- 다음 분의 NG가 5개 이상이면 NG=1
- 시간순으로 train/validation/test-NG/test-normal 분할
- scaler는 train에만 `fit`

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data/raw/melting_tank.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["MELT_TEMP", "MOTORSPEED", "MELT_WEIGHT"]
SEQUENCE_LENGTH = 10
QUICK_RUN = False

In [2]:
[FEATURES]

[['MELT_TEMP', 'MOTORSPEED', 'MELT_WEIGHT']]

In [3]:
[*FEATURES]

['MELT_TEMP', 'MOTORSPEED', 'MELT_WEIGHT']

In [4]:
df = pd.read_csv(DATA_PATH, usecols=["STD_DT", *FEATURES, "TAG"])
df.head()

,STD_DT,MELT_TEMP,MOTORSPEED,MELT_WEIGHT,TAG
0,2020-03-04 0:00,489,116,631,OK
1,2020-03-04 0:00,433,78,609,OK
2,2020-03-04 0:00,464,154,608,OK
3,2020-03-04 0:00,379,212,606,OK
4,2020-03-04 0:00,798,1736,604,OK


In [5]:
df["STD_DT"] = pd.to_datetime(df["STD_DT"], errors="raise")
df.head()

,STD_DT,MELT_TEMP,MOTORSPEED,MELT_WEIGHT,TAG
0,2020-03-04,489,116,631,OK
1,2020-03-04,433,78,609,OK
2,2020-03-04,464,154,608,OK
3,2020-03-04,379,212,606,OK
4,2020-03-04,798,1736,604,OK


In [6]:
df["target_ng"] = (df["TAG"] == "NG").astype("int8")
df = df.sort_values("STD_DT", kind="stable").reset_index(drop=True)
df.columns

Index(['STD_DT', 'MELT_TEMP', 'MOTORSPEED', 'MELT_WEIGHT', 'TAG', 'target_ng'], dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835200 entries, 0 to 835199
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   STD_DT       835200 non-null  datetime64[ns]
 1   MELT_TEMP    835200 non-null  int64         
 2   MOTORSPEED   835200 non-null  int64         
 3   MELT_WEIGHT  835200 non-null  int64         
 4   TAG          835200 non-null  object        
 5   target_ng    835200 non-null  int8          
dtypes: datetime64[ns](1), int64(3), int8(1), object(1)
memory usage: 32.7+ MB


In [8]:
assert not df[[*FEATURES, "target_ng"]].isna().any().any()
print("rows:", len(df), "minutes:", df["STD_DT"].nunique())

rows: 835200 minutes: 83520


In [9]:
df[:20]

,STD_DT,MELT_TEMP,MOTORSPEED,MELT_WEIGHT,TAG,target_ng
0,2020-03-04 00:00:00,489,116,631,OK,0
1,2020-03-04 00:00:00,433,78,609,OK,0
2,2020-03-04 00:00:00,464,154,608,OK,0
3,2020-03-04 00:00:00,379,212,606,OK,0
4,2020-03-04 00:00:00,798,1736,604,OK,0
5,2020-03-04 00:00:00,743,1722,603,OK,0
6,2020-03-04 00:00:00,390,212,602,OK,0
7,2020-03-04 00:00:00,493,152,600,OK,0
8,2020-03-04 00:00:00,427,0,599,OK,0
9,2020-03-04 00:00:00,489,148,598,OK,0


In [10]:
for i, (timestamp, group) in enumerate(df.groupby("STD_DT", sort=True)):
    if i >= 3:
        break

    print(f'{timestamp = }')
    print(len(group))
    display(group)
    # group[FEATURES].to_numpy(dtype="float32")
    print('*' * 100)
    print()

timestamp = Timestamp('2020-03-04 00:00:00')
10


,STD_DT,MELT_TEMP,MOTORSPEED,MELT_WEIGHT,TAG,target_ng
0,2020-03-04,489,116,631,OK,0
1,2020-03-04,433,78,609,OK,0
2,2020-03-04,464,154,608,OK,0
3,2020-03-04,379,212,606,OK,0
4,2020-03-04,798,1736,604,OK,0
5,2020-03-04,743,1722,603,OK,0
6,2020-03-04,390,212,602,OK,0
7,2020-03-04,493,152,600,OK,0
8,2020-03-04,427,0,599,OK,0
9,2020-03-04,489,148,598,OK,0


****************************************************************************************************

timestamp = Timestamp('2020-03-04 00:01:00')
10


,STD_DT,MELT_TEMP,MOTORSPEED,MELT_WEIGHT,TAG,target_ng
10,2020-03-04 00:01:00,507,128,596,OK,0
11,2020-03-04 00:01:00,408,66,595,OK,0
12,2020-03-04 00:01:00,474,138,594,OK,0
13,2020-03-04 00:01:00,358,201,592,OK,0
14,2020-03-04 00:01:00,740,1740,590,OK,0
15,2020-03-04 00:01:00,772,1729,588,OK,0
16,2020-03-04 00:01:00,424,195,586,OK,0
17,2020-03-04 00:01:00,460,158,585,OK,0
18,2020-03-04 00:01:00,440,0,584,OK,0
19,2020-03-04 00:01:00,504,133,582,OK,0


****************************************************************************************************

timestamp = Timestamp('2020-03-04 00:02:00')
10


,STD_DT,MELT_TEMP,MOTORSPEED,MELT_WEIGHT,TAG,target_ng
20,2020-03-04 00:02:00,474,135,581,OK,0
21,2020-03-04 00:02:00,446,67,580,OK,0
22,2020-03-04 00:02:00,487,161,578,OK,0
23,2020-03-04 00:02:00,393,205,577,OK,0
24,2020-03-04 00:02:00,740,1748,575,OK,0
25,2020-03-04 00:02:00,761,1721,574,OK,0
26,2020-03-04 00:02:00,383,200,572,OK,0
27,2020-03-04 00:02:00,483,167,571,OK,0
28,2020-03-04 00:02:00,435,90,570,OK,0
29,2020-03-04 00:02:00,456,111,568,OK,0


****************************************************************************************************



In [38]:
sequences, minute_targets, timestamps = [], [], []

for timestamp, group in df.groupby("STD_DT", sort=True):
    if len(group) != SEQUENCE_LENGTH:
        continue
    sequences.append(group[FEATURES].to_numpy(dtype="float32"))
    minute_targets.append(int(group["target_ng"].sum() >= 5))
    timestamps.append(np.datetime64(timestamp))

In [12]:
sequences[:3]

[array([[ 489.,  116.,  631.],
        [ 433.,   78.,  609.],
        [ 464.,  154.,  608.],
        [ 379.,  212.,  606.],
        [ 798., 1736.,  604.],
        [ 743., 1722.,  603.],
        [ 390.,  212.,  602.],
        [ 493.,  152.,  600.],
        [ 427.,    0.,  599.],
        [ 489.,  148.,  598.]], dtype=float32),
 array([[ 507.,  128.,  596.],
        [ 408.,   66.,  595.],
        [ 474.,  138.,  594.],
        [ 358.,  201.,  592.],
        [ 740., 1740.,  590.],
        [ 772., 1729.,  588.],
        [ 424.,  195.,  586.],
        [ 460.,  158.,  585.],
        [ 440.,    0.,  584.],
        [ 504.,  133.,  582.]], dtype=float32),
 array([[ 474.,  135.,  581.],
        [ 446.,   67.,  580.],
        [ 487.,  161.,  578.],
        [ 393.,  205.,  577.],
        [ 740., 1748.,  575.],
        [ 761., 1721.,  574.],
        [ 383.,  200.,  572.],
        [ 483.,  167.,  571.],
        [ 435.,   90.,  570.],
        [ 456.,  111.,  568.]], dtype=float32)]

In [13]:
sequences[0].shape

(10, 3)

In [14]:
len(sequences)

83520

In [15]:
len(minute_targets)

83520

In [16]:
timestamps[:3]

[np.datetime64('2020-03-04T00:00:00.000000'),
 np.datetime64('2020-03-04T00:01:00.000000'),
 np.datetime64('2020-03-04T00:02:00.000000')]

In [17]:
len(timestamps)

83520

In [18]:
X_current = np.asarray(sequences, dtype="float32")
y_current = np.asarray(minute_targets, dtype="int8")
times_current = np.asarray(timestamps)

In [19]:
X_current.shape, y_current.shape, times_current.shape

((83520, 10, 3), (83520,), (83520,))

In [20]:
## 현재 분의 시퀀스로 다음 분의 상태를 예측
X, y, times = X_current[:-1], y_current[1:], times_current[:-1]
print("X:", X.shape, "y:", y.shape, "NG ratio:", y.mean())

X: (83519, 10, 3) y: (83519,) NG ratio: 0.18883128389947199


In [21]:
## 분 단위 타깃 기준 월별 불량률 집계 코드
monthly_sequence_df = pd.DataFrame({
    "YEAR_MONTH": pd.to_datetime(times).to_period("M"),
    "target_ng": y
})

## 월별 집계
monthly_seq_summary = (
    monthly_sequence_df.groupby("YEAR_MONTH")
    .agg(
        total_minutes=("target_ng", "count"),
        normal_minutes=("target_ng", lambda x: (x == 0).sum()),
        ng_minutes=("target_ng", "sum"),
        ng_ratio=("target_ng", "mean"),
    )
    .reset_index()
)

monthly_seq_summary["ng_rate_pct"] = (monthly_seq_summary["ng_ratio"] * 100).round(2).astype(str) + "%"

display(monthly_seq_summary)


,YEAR_MONTH,total_minutes,normal_minutes,ng_minutes,ng_ratio,ng_rate_pct
0,2020-03,40320,27771,12549,0.311235,31.12%
1,2020-04,43199,39977,3222,0.074585,7.46%


In [22]:
## 분 단위 타깃 기준 월/주차별 불량률 집계 코드

## times와 y를 기반으로 데이터프레임 구성
df_seq = pd.DataFrame({"timestamp": pd.to_datetime(times), "target_ng": y})  #

## 연도, 월, 연간 주차(ISO Week), 월 기준 주차 컬럼 생성
df_seq["YEAR"] = df_seq["timestamp"].dt.year
df_seq["MONTH"] = df_seq["timestamp"].dt.month


## 월 내 주차 계산: (해당 일 - 1) // 7 + 1 (1~7일: 1주차, 8~14일: 2주차 ...)
df_seq["WEEK_OF_MONTH"] = (df_seq["timestamp"].dt.day - 1) // 7 + 1
df_seq["WEEK_LABEL"] = (
    df_seq["MONTH"].astype(str)
    + "월 "
    + df_seq["WEEK_OF_MONTH"].astype(str)
    + "주차"
)

## 3월과 4월 데이터 필터링 후 주차별 집계
weekly_summary = (
    df_seq[df_seq["MONTH"].isin([3, 4])]
    .groupby(["MONTH", "WEEK_OF_MONTH", "WEEK_LABEL"], as_index=False)
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        total_minutes=("target_ng", "count"),
        normal_minutes=("target_ng", lambda x: (x == 0).sum()),
        ng_minutes=("target_ng", "sum"),
        ng_ratio=("target_ng", "mean"),
    )
)

## 퍼센트(%) 포맷 컬럼 추가 및 가독성 정렬
weekly_summary["ng_rate_pct"] = (
    (weekly_summary["ng_ratio"] * 100).round(2).astype(str) + "%"
)

## 컬럼 순서 정리 및 출력
result_cols = [
    "WEEK_LABEL",
    "start_time",
    "end_time",
    "total_minutes",
    "normal_minutes",
    "ng_minutes",
    "ng_ratio",
    "ng_rate_pct",
]
display(weekly_summary[result_cols])

,WEEK_LABEL,start_time,end_time,total_minutes,normal_minutes,ng_minutes,ng_ratio,ng_rate_pct
0,3월 1주차,2020-03-04,2020-03-07 23:59:00,5760,5760,0,0.000000,0.0%
1,3월 2주차,2020-03-08,2020-03-14 23:59:00,10080,10080,0,0.000000,0.0%
2,3월 3주차,2020-03-15,2020-03-21 23:59:00,10080,8699,1381,0.137004,13.7%
3,3월 4주차,2020-03-22,2020-03-28 23:59:00,10080,2731,7349,0.729067,72.91%
4,3월 5주차,2020-03-29,2020-03-31 23:59:00,4320,501,3819,0.884028,88.4%
5,4월 1주차,2020-04-01,2020-04-07 23:59:00,10080,7173,2907,0.288393,28.84%
6,4월 2주차,2020-04-08,2020-04-14 23:59:00,10080,9765,315,0.031250,3.12%
7,4월 3주차,2020-04-15,2020-04-21 23:59:00,10080,10080,0,0.000000,0.0%
8,4월 4주차,2020-04-22,2020-04-28 23:59:00,10080,10080,0,0.000000,0.0%
9,4월 5주차,2020-04-29,2020-04-30 23:58:00,2879,2879,0,0.000000,0.0%


* **train (3월 전체)**: 40,320건 중 불량 12,549건 (불량률 약 31.1%)
* **validation (4월 1주차)**: 10,080건 중 불량 2,907건 (불량률 약 28.8%)
* **test_ng (4월 2주차)**: 10,080건 중 불량 315건 (불량률 약 3.1%)
* **test_normal (4월 3주차 이후)**: 23,039건 중 불량 0건 (불량률 0.0%)

In [23]:
times

array(['2020-03-04T00:00:00.000000', '2020-03-04T00:01:00.000000',
       '2020-03-04T00:02:00.000000', ..., '2020-04-30T23:56:00.000000',
       '2020-04-30T23:57:00.000000', '2020-04-30T23:58:00.000000'],
      shape=(83519,), dtype='datetime64[us]')

In [24]:
masks = {
    "train": times <= np.datetime64("2020-03-31T23:59:00"),
    "validation": (times > np.datetime64("2020-03-31T23:59:00")) & (times <= np.datetime64("2020-04-07T23:59:00")),
    "test_ng": (times > np.datetime64("2020-04-07T23:59:00")) & (times <= np.datetime64("2020-04-14T23:59:00")),
    "test_normal": times > np.datetime64("2020-04-14T23:59:00"),
}

In [25]:
masks

{'train': array([ True,  True,  True, ..., False, False, False], shape=(83519,)),
 'validation': array([False, False, False, ..., False, False, False], shape=(83519,)),
 'test_ng': array([False, False, False, ..., False, False, False], shape=(83519,)),
 'test_normal': array([False, False, False, ...,  True,  True,  True], shape=(83519,))}

In [26]:
if QUICK_RUN:
    ## 각 시간 구간 전체에서 고르게 추출하여 클래스와 분포 변화를 함께 확인한다.
    quick_masks = {}
    for name, mask in masks.items():
        indices = np.flatnonzero(mask)
        selected = indices[np.linspace(0, len(indices) - 1, min(4_000, len(indices)), dtype=int)]
        quick_mask = np.zeros(len(X), dtype=bool)
        quick_mask[selected] = True
        quick_masks[name] = quick_mask
    masks = quick_masks

In [27]:
summary = []

for name, mask in masks.items():
    assert mask.any(), f"{name} 분할이 비어 있습니다."
    summary.append({"split": name, "samples": int(mask.sum()), "ng": int(y[mask].sum()), "ng_ratio": float(y[mask].mean())})
    
pd.DataFrame(summary)

,split,samples,ng,ng_ratio
0,train,40320,12549,0.311235
1,validation,10080,2907,0.288393
2,test_ng,10080,315,0.031250
3,test_normal,23039,0,0.000000


In [28]:
split_order = ["train", "validation", "test_ng", "test_normal"]
X_splits = {name: X[mask] for name, mask in masks.items()}
y_splits = {name: y[mask] for name, mask in masks.items()}

In [29]:
X_splits

{'train': array([[[ 489.,  116.,  631.],
         [ 433.,   78.,  609.],
         [ 464.,  154.,  608.],
         ...,
         [ 493.,  152.,  600.],
         [ 427.,    0.,  599.],
         [ 489.,  148.,  598.]],
 
        [[ 507.,  128.,  596.],
         [ 408.,   66.,  595.],
         [ 474.,  138.,  594.],
         ...,
         [ 460.,  158.,  585.],
         [ 440.,    0.,  584.],
         [ 504.,  133.,  582.]],
 
        [[ 474.,  135.,  581.],
         [ 446.,   67.,  580.],
         [ 487.,  161.,  578.],
         ...,
         [ 483.,  167.,  571.],
         [ 435.,   90.,  570.],
         [ 456.,  111.,  568.]],
 
        ...,
 
        [[ 484.,  134.,  670.],
         [ 431.,   65.,  669.],
         [ 515.,  161.,  670.],
         ...,
         [ 484.,  172.,  669.],
         [ 426.,   85.,  670.],
         [ 468.,  139.,  669.]],
 
        [[ 456.,  138.,  668.],
         [ 450.,    0., 6688.],
         [ 492.,  159.,  668.],
         ...,
         [ 471.,  146.,  690.]

In [30]:
X_splits["train"]

array([[[ 489.,  116.,  631.],
        [ 433.,   78.,  609.],
        [ 464.,  154.,  608.],
        ...,
        [ 493.,  152.,  600.],
        [ 427.,    0.,  599.],
        [ 489.,  148.,  598.]],

       [[ 507.,  128.,  596.],
        [ 408.,   66.,  595.],
        [ 474.,  138.,  594.],
        ...,
        [ 460.,  158.,  585.],
        [ 440.,    0.,  584.],
        [ 504.,  133.,  582.]],

       [[ 474.,  135.,  581.],
        [ 446.,   67.,  580.],
        [ 487.,  161.,  578.],
        ...,
        [ 483.,  167.,  571.],
        [ 435.,   90.,  570.],
        [ 456.,  111.,  568.]],

       ...,

       [[ 484.,  134.,  670.],
        [ 431.,   65.,  669.],
        [ 515.,  161.,  670.],
        ...,
        [ 484.,  172.,  669.],
        [ 426.,   85.,  670.],
        [ 468.,  139.,  669.]],

       [[ 456.,  138.,  668.],
        [ 450.,    0., 6688.],
        [ 492.,  159.,  668.],
        ...,
        [ 471.,  146.,  690.],
        [ 437.,   79.,  669.],
        [ 488.,

In [31]:
scaler = MinMaxScaler()
n_features = len(FEATURES)
_ = scaler.fit(X_splits["train"].reshape(-1, n_features))

In [32]:
split_order

['train', 'validation', 'test_ng', 'test_normal']

In [33]:
X_splits['train'].shape

(40320, 10, 3)

In [34]:
X_splits['validation'].shape

(10080, 10, 3)

In [35]:
for name in split_order:
    values = X_splits[name]
    X_splits[name] = scaler.transform(values.reshape(-1, n_features)).reshape(values.shape).astype("float32")
    np.save(OUTPUT_DIR / f"X_{name}.npy", X_splits[name])
    np.save(OUTPUT_DIR / f"y_{name}.npy", y_splits[name])

In [36]:
joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")

metadata = {"features": FEATURES, "sequence_length": SEQUENCE_LENGTH, "target_rule": "next-minute-majority", "quick_run": QUICK_RUN}

(ARTIFACT_DIR / "preprocessing.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print("전처리 산출물 저장 완료")

전처리 산출물 저장 완료


## 확인 사항

최종 입력 모양은 `(표본 수, 10, 3)`입니다. 검증 및 테스트 데이터의 값이 0~1 범위를 벗어날 수 있는데, 이는 학습 데이터 범위 밖의 값이 들어왔다는 의미이며 오류가 아닙니다.